In [1]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

import pandas as pd
import plotly.graph_objects as go
from scipy.interpolate import interp1d



In [26]:
GVM = 192*1e-3 #ps/mm
L = 1 #mm
c_nm_ps = 299792.458          # nm/ps
c_um_ps = c_nm_ps / 1000.0 

def wideband(Omega, deltaL):
    return np.sinc(GVM*L*Omega/2)**2 * 4* np.sin(deltaL*Omega/(2*c_um_ps))**2
deltaLs = np.linspace(-500,500,500)
HOMwb = []

#coincidences for wideband case
for deltaL in deltaLs:
    result = sp.integrate.quad(lambda Omega: wideband(Omega, deltaL), -np.inf, np.inf)
    HOMwb.append(result[0])

fig = go.Figure(
    data=go.Scatter(x=deltaLs, y=HOMwb, mode="lines+markers", name="Wideband")
)
fig.update_layout(
    title="HOM Coincidences (Wideband) vs Path-Length Difference",
    xaxis_title="ΔL (um)",
    yaxis_title="Coincidences",
)
fig.show()


/var/folders/b2/kybxzmpd3s13c1h0hqv37c3w0000gn/T/ipykernel_55387/3941688857.py:13: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  result = sp.integrate.quad(lambda Omega: wideband(Omega, deltaL), -np.inf, np.inf)


# Part 2

In [ ]:
#Speed of light in convenient units
c_nm_ps = 299792.458          # nm/ps
c_um_ps = c_nm_ps / 1000.0    # um/ps

#Read NB filter transmission vs wavelength and convert to frequency (THz)
transmission_spec = pd.read_csv("LL01-810_Spectrum.txt", sep=r"\s+", header=3)
wl_nm = transmission_spec["Wavelength(nm)"].to_numpy()
T = transmission_spec["Transmission"].to_numpy()
freq_thz = c_nm_ps / wl_nm

#Define detuning Omega = f - f0 around the physical center wavelength (basically reshuffle data so that the x-axis is detuning from center frequency instead of wavelength)
lambda0_nm = 810.0
f0 = c_nm_ps / lambda0_nm
sort_idx = np.argsort(freq_thz)
freq_sorted = freq_thz[sort_idx]
T_sorted = T[sort_idx]
Omega_data = freq_sorted - f0

#Linear interpolation avoids overshoot artifacts without explicit clipping
spec_interp = interp1d(Omega_data, T_sorted, kind="linear", bounds_error=False, fill_value=0.0)

#Plot filter data and interpolation as a sanity check
go.Figure(data=go.Scatter(x=Omega_data, y=T_sorted, mode="markers", name="Data Points")).show()
frequency_grid = np.linspace(Omega_data.min(), Omega_data.max(), 10000)
fig = go.Figure(data=go.Scatter(x=frequency_grid, y=spec_interp(frequency_grid), mode="lines"))
fig.update_layout(title="Interpolated NB Transmission vs Detuning",xaxis_title="Omega (THz)",yaxis_title="Transmission",)
fig.show()


GVM = 192.0 * 1e-3  #Convert to ps/mm
L_mm = 1.0

def narrowband(Omega, deltaL_um):
    sinc_arg = 0.5 * GVM * L_mm * Omega
    sin_arg = 0.5 * deltaL_um * Omega / c_um_ps
    return spec_interp(Omega) * spec_interp(-Omega) * np.sinc(sinc_arg) ** 2 * 4.0 * np.sin(sin_arg) ** 2

#Integrate over the frequency span where the interpolated spectrum is defined (since it goes to zero outside that range, we can just integrate over the data range instead of all frequencies)
omega_lim = min(abs(Omega_data.min()), abs(Omega_data.max()))

#filtered coincidences are in a much wider range than in the wideband case because we are essentially restricting the allowed energies of the photons, so because of the uncertainty principle we are allowing a wider range of arrival times!
deltaLs = np.linspace(-6000, 6000, 500)
HOMnb = []

#Here we use trapezoid integration since quad was giving me some numerical artifacts due to the oscillatory behavior of the integrand. 
omega_int = np.linspace(-omega_lim, omega_lim, 20001)
base_weight = spec_interp(omega_int) * spec_interp(-omega_int) * np.sinc(0.5 * GVM * L_mm * omega_int) ** 2

for deltaL in deltaLs:
    sin_arg = 0.5 * deltaL * omega_int / c_um_ps
    integrand = base_weight * 4.0 * np.sin(sin_arg) ** 2
    HOMnb.append(np.trapezoid(integrand, omega_int))

fig = go.Figure(data=go.Scatter(x=deltaLs, y=HOMnb, mode="lines+markers", name="Narrowband"))
fig.update_layout(
    title="HOM Coincidences (Narrowband) vs Path-Length Difference",
    xaxis_title="ΔL (um)",
    yaxis_title="Coincidences",
)
fig.show()